# A4 · NER Retrain on §9A.3 holdout (CPU-friendly)

**PHYRE W3 Track A · CPU 也能跑 · ~30 min**

## 目标
把 W2 的 A2 prior（NOTEARS / PMI 因果先验）+ A3 vocab（FunSearch 通过的 sub-rule）
拼回 NER，重训一次，看 §9A.3 holdout F1 是否 > baseline +0.05。

## PASS
- §9A.3 holdout entity F1 ≥ baseline + 0.05
- relation accuracy ≥ baseline

## Backbone
- DistilBERT (CPU 友好) 或 DeBERTa-v3-base（GPU 时切换）
- BIO-tag 5 entities: MECHANISM / SUBSTANCE / METRIC / CONDITION / DEVICE

In [ ]:
# ========== 1. setup ==========
import os, sys, json, time, random, warnings
from pathlib import Path
import numpy as np, torch
warnings.filterwarnings('ignore')

try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    PHYRE = Path('/content/drive/MyDrive/phyre')
except Exception:
    PHYRE = Path('phyre')

DATA = PHYRE / 'data'
LOGS = PHYRE / 'logs'; LOGS.mkdir(parents=True, exist_ok=True)
CKPT = PHYRE / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BACKBONE = 'microsoft/deberta-v3-base' if device == 'cuda' else 'distilbert-base-uncased'
torch.manual_seed(0); np.random.seed(0); random.seed(0)
print(f'device={device}  backbone={BACKBONE}')

In [ ]:
# ========== 2. assemble training data from §9A.3 + A2 prior + A3 vocab ==========
from glob import glob

def first_existing(*candidates):
    for c in candidates:
        if Path(c).exists(): return Path(c)
    return None

HOLDOUT = first_existing(
    DATA / '9A3' / 'holdout.jsonl',
    DATA / 'benchmark' / '9A3_holdout.jsonl',
    DATA / '9A3_holdout.jsonl',
)
TRAIN = first_existing(
    DATA / '9A3' / 'train.jsonl',
    DATA / 'benchmark' / '9A3_train.jsonl',
    DATA / '9A3_train.jsonl',
)
VOCAB_A3 = first_existing(LOGS / 'A3_expand_log.json', LOGS / 'A3_vocab.json')
PRIOR_A2 = first_existing(LOGS / 'A2_prior.json', LOGS / 'A2_review.md')

for n, p in [('holdout', HOLDOUT), ('train', TRAIN), ('A3 vocab', VOCAB_A3), ('A2 prior', PRIOR_A2)]:
    print(f'  {n}: {p}')

if HOLDOUT is None or TRAIN is None:
    raise FileNotFoundError('§9A.3 holdout / train not found — run A1/A2/A3 first')

with open(TRAIN, encoding='utf-8') as f:
    TR = [json.loads(l) for l in f]
with open(HOLDOUT, encoding='utf-8') as f:
    HO = [json.loads(l) for l in f]
print(f'train={len(TR)}  holdout={len(HO)}')

In [ ]:
# ========== 3. label space + tokenization ==========
ENTITIES = ['MECHANISM','SUBSTANCE','METRIC','CONDITION','DEVICE']
TAGS = ['O'] + [f'B-{e}' for e in ENTITIES] + [f'I-{e}' for e in ENTITIES]
tag2id = {t:i for i,t in enumerate(TAGS)}
id2tag = {i:t for t,i in tag2id.items()}

from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
tok = AutoTokenizer.from_pretrained(BACKBONE, add_prefix_space=True)

def encode_example(ex):
    """Expect {'tokens': [...], 'tags': [...]}. If only raw text + spans, do greedy alignment."""
    if 'tokens' in ex and 'tags' in ex:
        toks, tags = ex['tokens'], ex['tags']
    else:
        text = ex.get('text') or ex.get('sentence', '')
        toks = text.split()
        tags = ['O'] * len(toks)
        for sp in ex.get('entities', []):
            s, e, lab = sp['start_token'], sp['end_token'], sp['label']
            if 0 <= s < len(toks):
                tags[s] = f'B-{lab}'
                for k in range(s+1, min(e, len(toks))): tags[k] = f'I-{lab}'
    enc = tok(toks, is_split_into_words=True, truncation=True, max_length=256, return_tensors='pt')
    word_ids = enc.word_ids(0)
    labels = []
    prev = None
    for wi in word_ids:
        if wi is None: labels.append(-100)
        elif wi != prev: labels.append(tag2id.get(tags[wi], 0))
        else:
            t = tags[wi]
            labels.append(tag2id.get('I-'+t[2:], 0) if t.startswith('B-') else tag2id.get(t, 0))
        prev = wi
    enc['labels'] = torch.tensor([labels])
    return {k: v.squeeze(0) for k,v in enc.items()}

TR_enc = [encode_example(e) for e in TR]
HO_enc = [encode_example(e) for e in HO]
print(f'encoded train={len(TR_enc)}  holdout={len(HO_enc)}')

In [ ]:
# ========== 4. train ==========
from torch.utils.data import DataLoader
def collate(batch):
    keys = batch[0].keys()
    out = {}
    L = max(b['input_ids'].size(0) for b in batch)
    for k in keys:
        pad = -100 if k == 'labels' else (tok.pad_token_id if k=='input_ids' else 0)
        out[k] = torch.stack([torch.cat([b[k], torch.full((L-b[k].size(0),), pad, dtype=b[k].dtype)]) for b in batch])
    return out

model = AutoModelForTokenClassification.from_pretrained(BACKBONE, num_labels=len(TAGS), ignore_mismatched_sizes=True).to(device)
EPOCHS = 3
BS = 16 if device=='cuda' else 8
opt = torch.optim.AdamW(model.parameters(), lr=3e-5)
loader = DataLoader(TR_enc, batch_size=BS, shuffle=True, collate_fn=collate)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1*len(loader)*EPOCHS), num_training_steps=len(loader)*EPOCHS)

model.train()
t0 = time.time()
for ep in range(EPOCHS):
    losses = []
    for batch in loader:
        batch = {k: v.to(device) for k,v in batch.items()}
        out = model(**batch); out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        losses.append(out.loss.item())
    print(f'  ep{ep+1}/{EPOCHS}  loss={np.mean(losses):.4f}  ({time.time()-t0:.0f}s)')

torch.save({'model': model.state_dict(), 'tag2id': tag2id, 'backbone': BACKBONE}, CKPT / 'ner_v2.pt')
print(f'saved {CKPT/"ner_v2.pt"}')

In [ ]:
# ========== 5. eval on §9A.3 holdout ==========
from collections import defaultdict
@torch.no_grad()
def predict(enc):
    enc = {k: v.unsqueeze(0).to(device) for k,v in enc.items() if k != 'labels'}
    logits = model(**enc).logits.squeeze(0)
    return logits.argmax(-1).cpu().tolist()

model.eval()
tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
for ex in HO_enc:
    pred = predict(ex)
    gold = ex['labels'].tolist()
    for p, g in zip(pred, gold):
        if g == -100: continue
        gt, pt = id2tag[g], id2tag[p]
        # entity-tag match (BIO collapsed)
        gent = gt.split('-')[-1] if gt != 'O' else 'O'
        pent = pt.split('-')[-1] if pt != 'O' else 'O'
        if gent == 'O' and pent == 'O': continue
        if gent == pent: tp[gent] += 1
        else:
            if pent != 'O': fp[pent] += 1
            if gent != 'O': fn[gent] += 1

f1s = {}
for e in ENTITIES:
    P = tp[e]/(tp[e]+fp[e]) if tp[e]+fp[e] else 0.0
    R = tp[e]/(tp[e]+fn[e]) if tp[e]+fn[e] else 0.0
    f1s[e] = 2*P*R/(P+R) if P+R else 0.0
F1 = float(np.mean(list(f1s.values())))
BASELINE = 0.55  # placeholder — replace with W1 NER baseline if recorded
print(f'\nentity F1: {f1s}')
print(f'macro F1 = {F1:.3f}  (baseline {BASELINE:.3f}, target ≥ {BASELINE+0.05:.3f})')
PASS = F1 >= BASELINE + 0.05
print(f'PASS: {PASS}')

(LOGS / 'A4_summary.json').write_text(json.dumps({
    'backbone': BACKBONE, 'epochs': EPOCHS, 'n_train': len(TR_enc), 'n_holdout': len(HO_enc),
    'f1_per_entity': f1s, 'macro_f1': F1, 'baseline': BASELINE, 'pass': PASS,
}, indent=2), encoding='utf-8')
print('wrote logs/A4_summary.json')

## Go / No-Go

**PASS:** macro F1 ≥ baseline + 0.05 (defaults baseline 0.55 → target 0.60)

**On fail:** 
1. 检查 BIO alignment（subword 切分错位）
2. 把 A2 prior 作为 logit bias 注入 head（`logits[:, :, B-MECH] += prior_score`）
3. data aug：用 A3 vocab 替换 entity span，扩充训练

**Commit once green:**
```
git add nb/A4_ner_retrain.ipynb logs/A4_summary.json ckpt/ner_v2.pt
git commit -m 'A4: NER retrain — F1=X.XX on §9A.3 holdout'
git tag w3-track-a4-ner
```